# NagarVani — midsem prototype
**Team:** _add every member's name here_ · AI course project (AI CP), VIT Pune

**Marathi voice-first triage for PMC grievances: speech → text → department → priority → route or defer to a human.**

### How to run (about 5 minutes)
1. `Runtime → Change runtime type → T4 GPU` → Save.
2. `Runtime → Run all`. Loading Whisper large-v3 takes 2–4 minutes the first time.
3. The last cell prints a public `https://….gradio.live` link. Open it, pick a ward, record or type a complaint.

### What this prototype is (say this honestly in the evaluation)
- One working end-to-end slice of the PBL-3 design: **M1 ingestion → M2 ASR → M3 normalisation (light) → M4a classifier → M5 severity rules → M7 routing / abstention → ticket table**.
- **Seed data:** two independently drafted sets, both AI-drafted and human-checked. **Set A**: 200 complaints (8 departments × 25). **Set B**: 260 complaints (the same 8 departments + *Other*, with severity labels). Devanagari, romanised and code-mixed Marathi, some Hindi. They stand in for the real corpus (D0) we are building next.
- **Honest held-out number:** Step 3b trains on one set and tests on the other, so the model is scored on sentences written separately from its training data.
- **Not built yet (planned):** M4b location extraction, M6 duplicate detection, calibration, officer dashboard, PMC CARE adapter, closure verification.

In [ ]:
# Step 0 — install (openai-whisper for speech, gradio for the web page)
!pip -q install -U openai-whisper gradio

## Step 1 — Config and seed dataset
*Viva note:* 8 departments mirroring PMC CARE categories, plus *Other* for complaints no department fits. 460 rows in total is a seed set; every number below is labelled **preliminary**.

In [ ]:
import os, re, uuid, unicodedata, datetime as dt
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from zoneinfo import ZoneInfo
IST = ZoneInfo("Asia/Kolkata")   # Colab servers run on UTC; tickets must show Pune time
OUT_DIR = "/content" if os.path.exists("/content") else "."
SEED = 42
MIN_LETTERS = 3   # fewer letters than this = nothing to classify
ASR_TAU = 0.45   # below this speech confidence, the complaint goes to human review

# PMC ward offices (verify spellings against the PMC website before the demo)
WARDS = ["Aundh-Baner", "Kothrud-Bavdhan", "Warje-Karvenagar", "Dhole Patil Road",
         "Hadapsar-Mundhwa", "Yerawada-Kalas-Dhanori", "Nagar Road-Wadgaonsheri",
         "Shivajinagar-Ghole Road", "Kasba-Vishrambaugwada", "Bhavani Peth", "Bibwewadi",
         "Sinhagad Road", "Dhankawadi-Sahakarnagar", "Kondhwa-Yewalewadi", "Wanowrie-Ramtekdi"]

DEPT_LABEL = {  # English / Marathi display names
    "roads": "Roads / पथ विभाग",
    "water_supply": "Water Supply / पाणीपुरवठा",
    "drainage": "Drainage / ड्रेनेज",
    "solid_waste": "Solid Waste / घनकचरा",
    "street_lights": "Street Lights / विद्युत (पथदिवे)",
    "trees_garden": "Trees & Garden / उद्यान व वृक्ष",
    "encroachment": "Encroachment & Building / अतिक्रमण व बांधकाम",
    "health_mosquito": "Health & Mosquito / आरोग्य व कीटक नियंत्रण",
    "other": "Other / इतर (officer to assign)",
}

DATA = {
 "roads": [
  "आमच्या गल्लीत रस्त्यावर मोठा खड्डा पडला आहे, गाड्या अडकत आहेत.",
  "सिंहगड रोडवर शाळेसमोर खूप खड्डे आहेत, मुलं पडत आहेत.",
  "रस्त्याचे काम अर्धवट सोडले आहे, दोन महिने झाले.",
  "फुटपाथ तुटलेला आहे, चालायला जागा नाही.",
  "पावसामुळे रस्ता पूर्ण खचला आहे.",
  "डांबरीकरण केल्यानंतर लगेच रस्ता उखडला.",
  "स्पीड ब्रेकर खूप उंच आहे आणि त्यावर पांढरे पट्टे नाहीत.",
  "चौकात रस्त्याच्या मधोमध खोल खड्डा आहे, रात्री दिसत नाही.",
  "रस्ता खोदून ठेवला आहे पण परत बुजवला नाही.",
  "आमच्या सोसायटीसमोरचा रस्ता पूर्णपणे खराब झाला आहे.",
  "दुभाजक तुटला आहे आणि लोखंडी सळ्या बाहेर आल्या आहेत.",
  "पुलावर रस्त्याला भेगा पडल्या आहेत.",
  "रस्त्यावरील पेव्हर ब्लॉक निघाले आहेत, दुचाकी घसरते.",
  "नवीन रस्ता बनवला पण एका आठवड्यात खड्डे पडले.",
  "गल्लीतला रस्ता कच्चा आहे, पावसात चिखल होतो.",
  "Rastyavar khup mothe khadde padle ahet, please lavkar durusti kara.",
  "Amchya galli madhla road puraa kharab zala ahe.",
  "Main road var pothole ahe, kal ek bike wala padla.",
  "रोडवर pothole आहे, रोज accident होत आहेत.",
  "footpath वरचे tiles तुटले आहेत, म्हाताऱ्या लोकांना त्रास होतो.",
  "सड़क पर बहुत बड़े गड्ढे हैं, कोई ध्यान नहीं दे रहा।",
  "हमारी गली की सड़क पूरी टूट गई है।",
  "road ka kaam aadha chhod diya hai, bahut dhool udti hai",
  "रस्त्याच्या कडेला खोल खड्डा खोदून ठेवला आहे, बॅरिकेड नाही.",
  "उड्डाणपुलाखालचा रस्ता उखडलेला आहे."
 ],
 "water_supply": [
  "तीन दिवसांपासून नळाला पाणी आलेले नाही.",
  "पाणी खूप कमी दाबाने येत आहे, टाकी भरत नाही.",
  "नळाला गढूळ आणि वास येणारे पाणी येत आहे.",
  "रस्त्यावर पाण्याची पाईपलाईन फुटली आहे, खूप पाणी वाया जात आहे.",
  "पाणीपुरवठ्याची वेळ रोज बदलते, कोणी सांगत नाही.",
  "आमच्या भागात टँकर पाठवा, पिण्यासाठी पाणी नाही.",
  "नवीन नळ कनेक्शनसाठी अर्ज केला पण अजून जोडणी झाली नाही.",
  "पाण्याचा व्हॉल्व्ह गळत आहे, दिवसभर पाणी वाहत असते.",
  "पिण्याच्या पाण्यात अळ्या दिसत आहेत.",
  "सकाळी फक्त अर्धा तास पाणी येते.",
  "पाणी मीटर तुटला आहे.",
  "आमच्या इमारतीला आठवडाभर पाणीपुरवठा बंद आहे.",
  "Nalala paani yet nahi, don divas zale.",
  "Paani khup ghaan yet ahe, pinyasarkha nahi.",
  "pipeline leak झाली आहे, रस्त्यावर पाणी वाहत आहे.",
  "water supply बंद आहे, tanker पाठवा please.",
  "low pressure मुळे वरच्या मजल्यावर पाणी पोहोचत नाही.",
  "पानी की सप्लाई तीन दिन से बंद है।",
  "नल में गंदा पानी आ रहा है।",
  "pani ka pressure bahut kam hai, tanki nahi bharti",
  "सार्वजनिक नळ तुटला आहे, पाणी वाया जात आहे.",
  "पाण्याच्या पाईपमधून हवा येते, पाणी नाही.",
  "पाणी पुरवठा रात्री उशिरा होतो, आम्हाला झोपेतून उठावे लागते.",
  "आमच्या वस्तीत पिण्याच्या पाण्याची मोठी टंचाई आहे.",
  "पाण्याच्या टाकीची स्वच्छता अनेक महिन्यांपासून केली नाही."
 ],
 "drainage": [
  "ड्रेनेज तुंबले आहे, घाण पाणी रस्त्यावर येत आहे.",
  "गटार उघडे आहे, खूप दुर्गंधी येते.",
  "चेंबरचे झाकण तुटले आहे, कोणी पडू शकते.",
  "सांडपाणी घरात शिरत आहे.",
  "पावसाळी पाण्याचा निचरा होत नाही, रस्त्यावर पाणी साचते.",
  "नाल्यात कचरा अडकला आहे, पाणी पुढे जात नाही.",
  "मॅनहोल उघडे आहे, रात्री खूप धोका आहे.",
  "ड्रेनेज लाइन फुटली आहे, सगळीकडे घाण पसरली आहे.",
  "सोसायटीच्या बाहेर गटाराचे पाणी वाहत आहे.",
  "नाल्याची साफसफाई पावसाळ्यापूर्वी झाली नाही.",
  "गटार भरून वाहत आहे, दुर्गंधी पसरली आहे.",
  "पावसात आमच्या गल्लीत गुडघाभर पाणी साचते.",
  "Drainage block zala ahe, ghaan paani road var yet ahe.",
  "Gutter overflow hot ahe, khup vaas yeto.",
  "chamber चे cover गायब आहे, लहान मुलं खेळतात तिथे.",
  "drainage चे पाणी घरात येत आहे, लवकर या.",
  "नाली जाम है, गंदा पानी सड़क पर बह रहा है।",
  "मैनहोल का ढक्कन टूटा हुआ है।",
  "gutter bahut badbu maar raha hai, saaf karwao",
  "पावसाळी गटारांची जाळी बंद झाली आहे.",
  "शौचालयाचे सांडपाणी उघड्यावर सोडले जात आहे.",
  "नाला भरून वाहत आहे, जवळच्या घरांना धोका आहे.",
  "ड्रेनेजचे काम केल्यानंतर चेंबर रस्त्यापेक्षा उंच राहिले आहे.",
  "गटाराची भिंत कोसळली आहे.",
  "भुयारी गटार योजनेचे काम अर्धवट आहे."
 ],
 "solid_waste": [
  "चार दिवसांपासून कचरा गाडी आलेली नाही.",
  "रस्त्याच्या कोपऱ्यावर कचऱ्याचा मोठा ढीग साचला आहे.",
  "कचराकुंडी भरून वाहत आहे, कुत्री कचरा पसरवतात.",
  "लोक मोकळ्या जागेत कचरा टाकतात, कोणी उचलत नाही.",
  "कचरा जाळला जात आहे, धूर घरात येतो.",
  "घंटागाडी वेळेवर येत नाही.",
  "बांधकामाचा राडारोडा रस्त्याच्या कडेला टाकला आहे.",
  "ओला आणि सुका कचरा एकत्र उचलला जातो.",
  "बाजाराजवळ सडलेल्या भाज्यांचा कचरा पडून आहे.",
  "मेलेले जनावर रस्त्यावर पडले आहे, उचलायला कोणी आले नाही.",
  "Kachra gaadi don divas aali nahi.",
  "Kachryacha dheeg khup motha zala ahe, vaas yeto.",
  "garbage उचलला नाही, संपूर्ण गल्लीत वास पसरला आहे.",
  "कचरा collection रोज होत नाही, please लक्ष द्या.",
  "dustbin तुटलेली आहे आणि कचरा रस्त्यावर पडतो.",
  "कूड़ा चार दिन से नहीं उठाया गया।",
  "कचरे का ढेर लगा है, बहुत बदबू आ रही है।",
  "kachra gaadi nahi aa rahi hai, bahut pareshani hai",
  "नदीकाठी लोक कचरा टाकत आहेत.",
  "सोसायटीचा कचरा घेण्यास कर्मचारी नकार देतात.",
  "रस्ता झाडला जात नाही, सगळीकडे प्लास्टिक पडले आहे.",
  "हॉटेलवाले रात्री रस्त्यावर कचरा टाकतात.",
  "शाळेच्या बाजूला कचऱ्याचा ढीग आहे.",
  "कचरा उचलल्यावर कुंडीभोवती घाण तशीच राहते.",
  "सार्वजनिक ठिकाणी कचराकुंडी नाही."
 ],
 "street_lights": [
  "आमच्या गल्लीतील पथदिवा एक आठवड्यापासून बंद आहे.",
  "रस्त्यावरचे सगळे दिवे रात्री बंद असतात, खूप अंधार असतो.",
  "पथदिवे दिवसाही चालू असतात, वीज वाया जाते.",
  "विजेचा खांब वाकला आहे, कधीही पडू शकतो.",
  "खांबावरची वायर लोंबकळत आहे, करंट लागू शकतो.",
  "उद्यानातील दिवे लागत नाहीत.",
  "पुलावरचा दिवा चालू-बंद होत राहतो.",
  "अंधारामुळे रात्री महिलांना चालायला भीती वाटते.",
  "नवीन पथदिवे लावण्याची गरज आहे, रस्ता पूर्ण अंधारात आहे.",
  "दिव्याच्या खांबाचे झाकण उघडे आहे, वायर बाहेर आहेत.",
  "Street light band ahe, galli madhe purna andhar asto.",
  "Divyacha khamb vakla ahe, dhoka ahe.",
  "street light चालू होत नाही, चोरीच्या घटना वाढल्या आहेत.",
  "light pole वरची wire तुटून खाली पडली आहे.",
  "सिग्नलजवळचे दिवे बंद आहेत.",
  "स्ट्रीट लाइट एक हफ्ते से बंद है।",
  "बिजली का खंभा झुक गया है, कभी भी गिर सकता है।",
  "street light kharab hai, raat ko bahut andhera rehta hai",
  "चौकातील हायमास्ट दिवा बंद आहे.",
  "पथदिव्याचा बल्ब फुटला आहे.",
  "दिव्याच्या खांबाला पावसात करंट येतो.",
  "स्मशानभूमीकडे जाणाऱ्या रस्त्यावर दिवे नाहीत.",
  "सोसायटीसमोरचे तीन दिवे बंद आहेत.",
  "LED दिवे लावले पण ते खूप मंद आहेत.",
  "बसथांब्याजवळ अंधार आहे, दिवा लावा."
 ],
 "trees_garden": [
  "वादळामुळे मोठे झाड रस्त्यावर पडले आहे.",
  "झाडाची फांदी विजेच्या तारांवर लोंबकळत आहे.",
  "झाड धोकादायकरीत्या वाकले आहे, कधीही पडू शकते.",
  "उद्यानातील खेळणी तुटलेली आहेत, मुलांना इजा होऊ शकते.",
  "बागेत गवत खूप वाढले आहे, साप दिसतात.",
  "झाडांची छाटणी करायची आहे, फांद्या घरात येत आहेत.",
  "कोणीतरी बेकायदेशीरपणे झाडे तोडत आहे.",
  "उद्यान वेळेवर उघडले जात नाही.",
  "पडलेले झाड दोन दिवसांपासून तसेच पडून आहे.",
  "उद्यानातील पाण्याचे कारंजे बंद आहे.",
  "Jhaad rastyavar padla ahe, traffic band ahe.",
  "Jhadachi phandi tutli ahe, gaadivar padu shakte.",
  "garden मधली swings तुटली आहेत.",
  "tree trimming ची गरज आहे, फांद्या खिडकीत येतात.",
  "पेड़ सड़क पर गिर गया है, रास्ता बंद है।",
  "पार्क में झूले टूटे हुए हैं।",
  "ped ki daali bahut neeche latak rahi hai, bus se takrati hai",
  "उद्यानातील वॉकिंग ट्रॅक खराब झाला आहे.",
  "रस्त्याकडेच्या झाडांना पाणी घातले जात नाही, झाडे सुकत आहेत.",
  "झाडाच्या मुळांमुळे भिंतीला तडे गेले आहेत.",
  "जुने वाळलेले झाड शाळेच्या आवारात आहे, धोका आहे.",
  "बागेतील बाकडे तुटले आहेत.",
  "उद्यानाची संरक्षक भिंत पडली आहे.",
  "वृक्ष प्राधिकरणाकडे परवानगी मागितली पण उत्तर नाही.",
  "झाडाची मोठी फांदी पार्क केलेल्या गाडीवर पडली."
 ],
 "encroachment": [
  "फुटपाथवर फेरीवाल्यांनी अतिक्रमण केले आहे, चालायला जागा नाही.",
  "शेजाऱ्याने रस्त्यावर बेकायदेशीर बांधकाम केले आहे.",
  "दुकानदारांनी सामान रस्त्यावर ठेवले आहे.",
  "मोकळ्या भूखंडावर अनधिकृत झोपड्या उभारल्या जात आहेत.",
  "इमारतीचे बेकायदेशीर वाढीव मजले बांधले जात आहेत.",
  "रस्त्यावर बेवारस गाड्या अनेक महिने उभ्या आहेत.",
  "होर्डिंग विनापरवाना लावले आहे.",
  "हॉटेलने फुटपाथवर टेबल लावले आहेत.",
  "सोसायटीच्या गेटसमोर हातगाडीवाले उभे राहतात.",
  "रात्री उशिरापर्यंत बांधकाम चालू असते, परवानगी आहे का तपासा.",
  "Footpath var feriwalyanni kabja kela ahe.",
  "Anadhikrut bandhkam chalu ahe, karvai kara.",
  "illegal construction सुरू आहे, कोणतीही परवानगी नाही.",
  "road वर दुकानांचे अतिक्रमण वाढले आहे.",
  "फुटपाथ पर ठेले वालों ने कब्जा कर लिया है।",
  "अवैध निर्माण चल रहा है, कार्रवाई करें।",
  "illegal hoarding laga hai, bahut khatarnak hai",
  "धोकादायक जुनी इमारत कधीही कोसळू शकते.",
  "सार्वजनिक जागेवर मंडप परवानगीशिवाय उभारला आहे.",
  "शेजाऱ्याने नाल्यावर बांधकाम केले आहे.",
  "रस्त्याच्या कडेला भंगार साठवले आहे.",
  "पार्किंगच्या जागेत दुकान थाटले आहे.",
  "बसथांब्यावर टपरी उभारली आहे.",
  "इमारतीचा भाग कोसळण्याच्या स्थितीत आहे.",
  "विनापरवाना फ्लेक्स बॅनर सगळीकडे लावले आहेत."
 ],
 "health_mosquito": [
  "आमच्या भागात खूप डास झाले आहेत, धूर फवारणी करा.",
  "साचलेल्या पाण्यात डासांच्या अळ्या दिसत आहेत.",
  "परिसरात डेंग्यूचे रुग्ण वाढले आहेत.",
  "मोकाट कुत्र्यांचा त्रास वाढला आहे, मुलांना चावतात.",
  "सार्वजनिक शौचालय खूप घाण आहे, स्वच्छता होत नाही.",
  "उघड्यावर मांस विक्री होत आहे, खूप माशा आहेत.",
  "औषध फवारणी अनेक महिन्यांपासून झाली नाही.",
  "भटक्या कुत्र्याने मुलाला चावले.",
  "उंदरांचा खूप त्रास आहे.",
  "शौचालयात पाणी आणि दिवा नाही.",
  "Dasancha khup tras ahe, fogging kara.",
  "Mokat kutre khup ahet, ratri bhunktat ani chavtat.",
  "fogging झालेली नाही, dengue चे cases वाढत आहेत.",
  "public toilet खूप dirty आहे.",
  "मच्छर बहुत हो गए हैं, फॉगिंग करवाइए।",
  "आवारा कुत्तों का बहुत आतंक है।",
  "dengue ke case badh rahe hain, spray karwao",
  "मोकळ्या प्लॉटमध्ये पाणी साचून डास वाढले आहेत.",
  "रस्त्यावर डुकरांचा वावर वाढला आहे.",
  "हॉटेलच्या मागे अन्न सडत आहे, उंदीर वाढले आहेत.",
  "सार्वजनिक शौचालयाची दुरवस्था झाली आहे.",
  "पिसाळलेला कुत्रा परिसरात फिरत आहे.",
  "भटक्या जनावरांमुळे रस्त्यावर अपघात होतात.",
  "मलेरियाचे रुग्ण आढळले आहेत, तपासणी करा.",
  "डासांमुळे लहान मुले आजारी पडत आहेत."
 ]
}



DATA_B = [  # [text, department, severity label]
 ["आमच्या रस्त्यावर मोठा खड्डा पडला आहे गेल्या दोन महिन्यांपासून कोणी बघत नाही", "roads", "P3"],
 ["रस्ता पूर्ण तुटलेला आहे पावसाळ्यात पाणी साचतं आणि गाड्या अडकतात", "roads", "P3"],
 ["फुटपाथ उखडलेला आहे बाळंबा चौकाजवळ लोक रस्त्यावर चालतात", "roads", "P3"],
 ["road la khup khadde padle aahet varje malwadi road var", "roads", "P3"],
 ["रस्ता खराब आहे divider तुटलेला आहे SB road वर", "roads", "P3"],
 ["पावसाळ्यात रस्त्यावर मोठमोठे खड्डे भरतात पाणी साचतं चालता येत नाही", "roads", "P3"],
 ["bridge खाली road settle झालाय cars ला problem होतोय", "roads", "P3"],
 ["आमच्या गल्लीतला रस्ता गेल्या पाच वर्षांपासून दुरुस्त केला नाही", "roads", "P4"],
 ["sadak todli aahe aani cement udhalay sagla", "roads", "P3"],
 ["speed breaker तुटलेला आहे अपघात होतोय", "roads", "P2"],
 ["शाळेजवळ रस्त्यात मोठा खड्डा आहे मुलांना धोका आहे", "roads", "P2"],
 ["divider तोडून लोक wrong side ने येतात accident होतोय", "roads", "P2"],
 ["नवीन रस्ता केला पण एका महिन्यातच खराब झाला", "roads", "P4"],
 ["manhole cover road var missing aahe khup dangerous aahe", "roads", "P1"],
 ["tar road ची surface पूर्ण उडालीय potholes मध्ये पाणी भरतंय", "roads", "P3"],
 ["signal light बंद आहे तीन दिवसांपासून crossing वर confusion होतंय", "roads", "P2"],
 ["रस्ता इतका खराब आहे की ambulance ला यायला problem होतोय", "roads", "P2"],
 ["footpath var hawkers बसलेत pedestrians ला चालता येत नाही", "roads", "P4"],
 ["cement road तुटलेला आहे dust उडतो allergies होतात", "roads", "P3"],
 ["आमच्या colony च्या entrance चा रस्ता पूर्ण खचलाय", "roads", "P3"],
 ["rasta khup kharab aahe gaadi chalvata yet nahi bikes la accident hoto", "roads", "P2"],
 ["waterlogging होतं रस्त्यावर प्रत्येक पावसात गुडघाभर पाणी", "roads", "P3"],
 ["रस्त्यावरचा मॅनहोल उघडा आहे कोणीतरी पडणार", "roads", "P1"],
 ["road construction चं काम अर्धवट सोडून दिलंय traffic ला problem", "roads", "P3"],
 ["पूल जवळचा रस्ता खचलाय heavy vehicles मुळे", "roads", "P2"],
 ["गल्लीत streetlight नाही अंधारात खड्ड्यांमध्ये पडतो", "roads", "P3"],
 ["नाल्याजवळचा रस्ता धुवून गेलाय पावसामुळे", "roads", "P3"],
 ["raste var water pipeline phodli aani road khodla tasa sodla", "roads", "P3"],
 ["आमच्या भागात पाणी येत नाही गेल्या तीन दिवसांपासून", "water_supply", "P2"],
 ["पाण्याचा pressure खूपच कमी आहे दुसऱ्या मजल्यावर पाणीच येत नाही", "water_supply", "P3"],
 ["pipeline leak झाला आहे रोज हजारो लिटर पाणी वाया जातंय", "water_supply", "P2"],
 ["paani ghadhalela yeta aani vaas yeto dudh sarkha dista", "water_supply", "P2"],
 ["water supply चा timing बदललाय कोणालाही माहित नाही", "water_supply", "P3"],
 ["पाण्याच्या टाकीला गळती लागलीय रोड वर पाणी वाहतंय", "water_supply", "P3"],
 ["main pipeline फुटली आहे पूर्ण रस्ता पाण्याने भरलाय", "water_supply", "P2"],
 ["आमच्या area ला alternate day पाणी येतं पण schedule सांगत नाहीत", "water_supply", "P3"],
 ["बोअरवेल खराब आहे गेल्या एका आठवड्यापासून पाणी नाही", "water_supply", "P2"],
 ["pani yellowish yeta filter karun pan piता yet nahi", "water_supply", "P2"],
 ["meter खराब आहे bill जास्त येतंय पण पाणी कमी येतंय", "water_supply", "P4"],
 ["water tanker मागवला पण आलाच नाही तीन दिवस झाले", "water_supply", "P2"],
 ["पूर्ण colony ला पाणी बंद आहे लोकांना खूप त्रास होतोय", "water_supply", "P2"],
 ["जुन्या pipeline मधून गंजलेलं पाणी येतंय आरोग्याला धोका", "water_supply", "P2"],
 ["water tank overflow होतो रोज रात्री पाणी वाया जातं", "water_supply", "P3"],
 ["valve तुटलेला आहे रस्त्यावर पाणी वाहत आहे", "water_supply", "P3"],
 ["amchya building la aathavdyatun ekdach paani yeta khup tras", "water_supply", "P2"],
 ["पाण्याच्या connection साठी apply केलंय सहा महिने झाले काहीच response नाही", "water_supply", "P4"],
 ["pipeline repair चं काम केलं पण परत leak होतंय", "water_supply", "P3"],
 ["पावसाळ्यात पाणी गढूळ येतंय chlorine चा वास येतो", "water_supply", "P3"],
 ["नळाला पाणी नाही येत सकाळपासून संध्याकाळपर्यंत", "water_supply", "P3"],
 ["drinking water chi pipe sanitary line jval futli aahe contamination hoil", "water_supply", "P1"],
 ["गटार तुंबलंय पाणी रस्त्यावर वाहतंय दुर्गंधी येतेय", "drainage", "P2"],
 ["drainage line तुटली आहे सांडपाणी घरात शिरतंय", "drainage", "P1"],
 ["गटाराचं पाणी ओव्हरफ्लो होतंय रस्त्यावर आणि घरांसमोर", "drainage", "P2"],
 ["nali tumbli aahe pavasalya madhe ghari paani yet astay", "drainage", "P2"],
 ["सांडपाणी रस्त्यावर वाहत आहे मुलांना शाळेत जाता येत नाही", "drainage", "P2"],
 ["manhole cover ओव्हरफ्लो होतंय दुर्गंधी असह्य आहे", "drainage", "P2"],
 ["drainage system पूर्णच fail झालंय पावसाळ्यात flooding होतं", "drainage", "P2"],
 ["गटारीचं पाणी विहिरीत मिसळतंय health hazard आहे", "drainage", "P1"],
 ["छोट्या नाल्याला कचरा अडकलाय पाणी वाहत नाही", "drainage", "P3"],
 ["sewage line choked aahe building samor paani saachlay", "drainage", "P2"],
 ["नवीन drainage line टाकायला हवी जुनी पूर्ण खराब झालीय", "drainage", "P3"],
 ["gutter tutlela aahe footpath var sandpaani vaahte", "drainage", "P2"],
 ["पावसाळ्यात नाल्याला पूर आलाय आजूबाजूच्या घरात पाणी शिरतंय", "drainage", "P2"],
 ["drainage चं cover गायब आहे रात्री कोणी पडेल", "drainage", "P1"],
 ["septic tank overflow होतंय दुर्गंधी सगळीकडे पसरतेय", "drainage", "P2"],
 ["hospital जवळ गटार तुंबलंय patients ना त्रास होतोय", "drainage", "P2"],
 ["आमच्या गल्लीत open drain आहे मुलांना धोका आहे", "drainage", "P2"],
 ["नाल्यात प्लास्टिक अडकलंय पाणी वाहत नाही तुंबतंय", "drainage", "P3"],
 ["rain water drainage बंद आहे पूर्ण area waterlog होतो", "drainage", "P2"],
 ["गटाराच्या cleaning साठी कितीदा तक्रार केली पण कोणी येत नाही", "drainage", "P3"],
 ["sandpaani ghari shirat aahe khup divas jhale koni baghina yet nahi", "drainage", "P2"],
 ["underground drainage pipe फुटली आहे रस्ता खचतोय", "drainage", "P2"],
 ["कचरा उचलला जात नाही गेल्या एका आठवड्यापासून", "solid_waste", "P3"],
 ["कचरागाडी येतच नाही आमच्या गल्लीत रोज", "solid_waste", "P3"],
 ["रस्त्यावर कचऱ्याचा ढीग लागलाय दुर्गंधी येतेय", "solid_waste", "P3"],
 ["kachara gadi yetach nahi weekly ekda pan nahi", "solid_waste", "P3"],
 ["dustbin overflow होतंय कोणी clean करत नाही", "solid_waste", "P3"],
 ["construction debris रस्त्यावर टाकलंय कोणीही उचलत नाही", "solid_waste", "P3"],
 ["शाळेजवळ कचऱ्याचा ढीग आहे मुलांच्या आरोग्याला धोका", "solid_waste", "P2"],
 ["dead animal रस्त्यावर पडलंय कोणी उचलत नाही दुर्गंधी", "solid_waste", "P2"],
 ["garden मध्ये कचरा टाकतात लोक पण सफाई होत नाही", "solid_waste", "P3"],
 ["hospital jval kacharyacha dhig aahe health risk aahe", "solid_waste", "P2"],
 ["wet waste आणि dry waste वेगळा collection होत नाही", "solid_waste", "P4"],
 ["रात्री लोक कचरा रस्त्यावर फेकतात गल्ली अस्वच्छ आहे", "solid_waste", "P3"],
 ["कचऱ्याचा ढीग जळवतात धूर येतो श्वसनाचा त्रास होतो", "solid_waste", "P2"],
 ["community bin तुटलेलं आहे कचरा बाहेर पडतो", "solid_waste", "P3"],
 ["plastic waste जाळतात नाल्याजवळ pollution होतंय", "solid_waste", "P2"],
 ["kachara vehicle chya timing badlalya koni sangat nahi", "solid_waste", "P4"],
 ["biomedical waste रस्त्यावर टाकलंय clinic जवळ", "solid_waste", "P2"],
 ["मोकळ्या जागेवर कचरा टाकतात dumping ground बनवलंय", "solid_waste", "P3"],
 ["कचरा वेचणाऱ्यांनी कचरा विखुरलाय रस्त्यावर", "solid_waste", "P3"],
 ["e-waste collection chi vyavastha nahi society madhe", "solid_waste", "P4"],
 ["kachara gaadi roj yet nahi alternate day pan nahi weekly pan nahi", "solid_waste", "P3"],
 ["garden waste उचललं जात नाही झाडांचा कचरा पडून आहे", "solid_waste", "P3"],
 ["आमच्या गल्लीतला streetlight बंद आहे अंधार आहे रात्री", "street_lights", "P3"],
 ["तीन streetlights बंद आहेत मुख्य रस्त्यावर अंधार आहे", "street_lights", "P3"],
 ["light pole वाकलेला आहे पडेल कोणावर तरी", "street_lights", "P2"],
 ["street light cha wire tutla aahe current lagel konala tari", "street_lights", "P1"],
 ["LED light बसवलं पण एका आठवड्यातच बंद पडलं", "street_lights", "P4"],
 ["शाळेजवळचा streetlight बंद आहे मुलांना अंधारात जावं लागतं", "street_lights", "P3"],
 ["रात्री अंधारात चोऱ्या होतात streetlight लावा", "street_lights", "P3"],
 ["timer var light chalte din savla pan light bandi aste", "street_lights", "P4"],
 ["mercury light ची बदली LED ने करायला हवी", "street_lights", "P4"],
 ["high mast light बंद आहे मैदानावर अंधार आहे", "street_lights", "P3"],
 ["light pole la rust laglay padel asa vatay dangerous aahe", "street_lights", "P2"],
 ["नवीन colony मध्ये streetlight नाही लावले अजून", "street_lights", "P3"],
 ["wire tutlela aahe khamb var latkat aahe shock basil", "street_lights", "P1"],
 ["दिवसा streetlight चालू असतो वीज वाया जाते timer लावा", "street_lights", "P4"],
 ["गल्लीत एकही दिवा नाही पूर्ण अंधार आहे", "street_lights", "P3"],
 ["pole var transformer chi problem aahe area la light nahi", "street_lights", "P2"],
 ["रस्त्यावर light नाही ladies ना रात्री जाता येत नाही safety issue", "street_lights", "P3"],
 ["streetlight maintenance होत नाही वारंवार तक्रार करूनही", "street_lights", "P3"],
 ["बाग मधलं मोठं झाड पडायला आलंय कोणावर तरी पडेल", "trees_garden", "P2"],
 ["झाड रस्त्यावर पडलंय traffic block झालंय", "trees_garden", "P2"],
 ["garden ची maintenance होत नाही गवत वाढलंय कचरा पडलाय", "trees_garden", "P4"],
 ["zhadachi fandi tutun padli aahe rasta block aahe", "trees_garden", "P2"],
 ["पार्कमधली बाकं तुटलेली आहेत बसता येत नाही", "trees_garden", "P4"],
 ["मोठं झाड वाकलंय electric wire वर पडेल", "trees_garden", "P1"],
 ["garden मध्ये mosquito breeding होतंय पाणी साचलंय", "trees_garden", "P3"],
 ["community garden ला कुंपण नाही जनावरं येतात", "trees_garden", "P4"],
 ["रस्त्यावरच्या झाडांची छाटणी करा branch तुटतात वाऱ्याने", "trees_garden", "P3"],
 ["park madhe swings tutalya aahet mulanna khelata yet nahi", "trees_garden", "P4"],
 ["बागेतला fountain बंद आहे पाणी गंधाळलेलं आहे", "trees_garden", "P4"],
 ["unauthorized tree cutting होतंय बांधकामासाठी", "trees_garden", "P3"],
 ["झाड कोसळलंय गाडीवर नुकसान झालंय", "trees_garden", "P2"],
 ["garden chi boundary wall tutli aahe", "trees_garden", "P4"],
 ["पावसाळ्यात मोठमोठी झाडं पडतात trimming करा", "trees_garden", "P3"],
 ["garden मध्ये drug addicts बसतात unsafe वाटतं", "trees_garden", "P3"],
 ["फुलझाडं तोडली कोणीतरी park मध्ये", "trees_garden", "P4"],
 ["dead tree khambासारखं उभं आहे पडेल रस्त्यावर", "trees_garden", "P2"],
 ["park madhe benches tutalya replace karayla havya", "trees_garden", "P4"],
 ["मोठं झाड घरावर पडायला आलंय मुळं उघडी पडलीत", "trees_garden", "P2"],
 ["अनधिकृत बांधकाम चालू आहे रात्री काम करतात", "encroachment", "P3"],
 ["फुटपाथवर दुकानं मांडलीत चालता येत नाही", "encroachment", "P3"],
 ["illegal construction चालू आहे मंजुरी न घेता building बांधतोय", "encroachment", "P3"],
 ["footpath var hawkers basle aahet jagaच nahi milat chalayला", "encroachment", "P3"],
 ["रस्त्यावर stall लावलाय traffic ला अडथळा होतोय", "encroachment", "P3"],
 ["शेजाऱ्यांनी common passage वर बांधकाम केलंय", "encroachment", "P3"],
 ["government जागेवर अतिक्रमण झालंय fence घातलंय", "encroachment", "P3"],
 ["parking area var unauthorized दुकान बांधलंय", "encroachment", "P3"],
 ["नाल्यावर अतिक्रमण झालंय बांधकाम केलंय drainage blocked", "encroachment", "P2"],
 ["footpath encroach केलंय dividers वर hawkers बसतात", "encroachment", "P3"],
 ["building ला permission नाही extra floor बांधतोय illegally", "encroachment", "P3"],
 ["mandirachi jagaa encroach keli government plot var ghare baandhli", "encroachment", "P3"],
 ["fire exit block केलंय building मध्ये storage ठेवलंय", "encroachment", "P1"],
 ["रस्ता अरुंद झालाय दोन्ही बाजूने अतिक्रमण", "encroachment", "P3"],
 ["playground वर construction material ठेवलंय मुलांना खेळता येत नाही", "encroachment", "P3"],
 ["illegal boarding चालवतोय residential area मध्ये", "encroachment", "P3"],
 ["jhopdi bandhli aahe rasta rokun traffic problem hoto", "encroachment", "P3"],
 ["building ने footprint पेक्षा जास्त बांधलंय शेजाऱ्यांना light येत नाही", "encroachment", "P3"],
 ["commercial use residential area मध्ये करतोय noise pollution", "encroachment", "P3"],
 ["wall todun rasta block kela aahe atikraman aahe", "encroachment", "P3"],
 ["मच्छर खूप आहेत डेंग्यूचा धोका आहे", "health_mosquito", "P2"],
 ["डासांचं प्रमाण वाढलंय fogging करा area मध्ये", "health_mosquito", "P2"],
 ["उघड्यावर मांस विक्री होतेय unhygienic आहे", "health_mosquito", "P3"],
 ["machhar khup aahet dengue pasarlay area madhe fogging kara", "health_mosquito", "P2"],
 ["stale food विकतात रस्त्यावर food poisoning होतं", "health_mosquito", "P2"],
 ["डासांचं breeding होतंय construction site वर पाणी साचलंय", "health_mosquito", "P2"],
 ["सार्वजनिक शौचालय अत्यंत घाणेरडं आहे सफाई होत नाही", "health_mosquito", "P3"],
 ["water tanker चं पाणी contaminated आहे लोक आजारी पडतोय", "health_mosquito", "P1"],
 ["रस्त्यावर थुंकतात पान खाऊन सगळीकडे विड्याची पिचकारी", "health_mosquito", "P4"],
 ["rats chi problem aahe area madhe plague sarkha hoyil", "health_mosquito", "P2"],
 ["hospital जवळ garbage ठेवलंय infection पसरतंय", "health_mosquito", "P2"],
 ["public toilet बंद आहे लोक उघड्यावर जातात", "health_mosquito", "P3"],
 ["मासळी बाजारात दुर्गंधी येतेय सफाई होत नाही", "health_mosquito", "P3"],
 ["dog bite cases वाढलेत area मध्ये stray dogs खूप आहेत", "health_mosquito", "P2"],
 ["fogging schedule माहित नाही dengue cases वाढतोय area मध्ये", "health_mosquito", "P2"],
 ["nali madhe mosquito breeding hotay saachlelya paanyat", "health_mosquito", "P2"],
 ["community hall मध्ये rats आहेत function करता येत नाही", "health_mosquito", "P3"],
 ["शाळेजवळ उघड्यावर मांस कापतात मुलांना बघावं लागतं unhygienic", "health_mosquito", "P3"],
 ["stray dogs aggressive झालेत मुलांना भीती वाटते", "health_mosquito", "P2"],
 ["vihar ghar jval kachra padlay swachata karnyat yetey nahi", "health_mosquito", "P3"],
 ["औषध फवारणी करा मच्छर खूप आहेत ताप आलाय लोकांना", "health_mosquito", "P2"],
 ["उंदीर खूप आहेत दुकानात अन्नपदार्थ खराब करतात", "health_mosquito", "P3"],
 ["काहीतरी problem आहे आमच्या area मध्ये बघा जरा", "other", "P4"],
 ["complaint करायची आहे कोणाला सांगायचं माहित नाही", "other", "P4"],
 ["tax office चा response येत नाही application pending आहे", "other", "P4"],
 ["amchya area madhe problem aahe please bghaa", "other", "P4"],
 ["birth certificate मिळालं नाही apply केलंय बराच वेळ झाला", "other", "P4"],
 ["property tax चा bill चुकीचा आलाय rectify करा", "other", "P4"],
 ["सरकारी कार्यालयातले कर्मचारी नीट बोलत नाहीत", "other", "P4"],
 ["building permission साठी apply केलंय status माहित नाही", "other", "P4"],
 ["noise pollution होतंय रात्री DJ लावतात", "other", "P3"],
 ["आमच्या area मध्ये बरंच काही problem आहे रस्ता पाणी कचरा सगळं", "other", "P3"],
 ["rasta ani drainage doghi problem aahe ek saath solve karaa", "other", "P3"],
 ["पाणी नाही कचरा उचलत नाहीत रस्ते खराब सगळंच बिघडलंय", "other", "P3"],
 ["khup tras hotoy amchya area madhe sagla kharab aahe", "other", "P4"],
 ["काय सांगू इथे सगळंच problem आहे", "other", "P4"],
 ["municipality काही करत नाही complaint दिली तरी", "other", "P4"],
 ["parking chi vyavastha nahi konich baghina yet nahi", "other", "P4"],
 ["रस्त्यावर गाड्या park करतात ambulance ला जागा नाही", "roads", "P2"],
 ["पावसामुळे रस्ता वाहून गेलाय माती आलीय", "roads", "P3"],
 ["navi pipeline takli ani rasta tasa sodla khodun", "water_supply", "P3"],
 ["गटार साफ करा पावसाळ्यापूर्वी नाहीतर flooding होईल", "drainage", "P3"],
 ["कचरा गाडी सकाळी खूप लवकर येते कोणाला माहीतच नसतं", "solid_waste", "P4"],
 ["halogen light लावली पण खूपच dim आहे काहीच दिसत नाही", "street_lights", "P4"],
 ["park madhe nali vaahte mulanna khelata yet nahi", "trees_garden", "P3"],
 ["footpath var auto rickshaw park kartaat chalata yet nahi pedestrian la", "encroachment", "P3"],
 ["मलेरिया पसरतोय area मध्ये action घ्या लवकर", "health_mosquito", "P2"],
 ["amchya bhagat khup problem aahet describe karata yenaar nahi", "other", "P4"],
 ["शाळेसमोरचा रस्ता खड्ड्यांनी भरलाय मुलांना अपघाताचा धोका", "roads", "P2"],
 ["water pipeline burst झालीय main road वर traffic jam", "water_supply", "P2"],
 ["गटाराचं झाकण तुटलंय कोणी पडणार आतमध्ये", "drainage", "P1"],
 ["कचऱ्यामुळे नाला तुंबलाय पूर्ण area मध्ये पाणी भरलंय", "drainage", "P2"],
 ["garbage collection irregular आहे week मध्ये एकदाच येतात", "solid_waste", "P3"],
 ["दोन pole मधला wire तुटून लोंबतोय शॉक बसेल", "street_lights", "P1"],
 ["झाड पडलंय रस्त्यावर blocking traffic completely", "trees_garden", "P2"],
 ["building collapse होण्याची शक्यता आहे cracks आलेत भिंतीला", "encroachment", "P1"],
 ["dengue outbreak area madhe fogging tatkal karaa hospital full zhalay", "health_mosquito", "P1"],
 ["रस्ता आणि गटार दोन्ही खराब आहे पावसाळ्यात worst होतं", "other", "P3"],
 ["flyover खालचा रस्ता खराब आहे अंधार असतो रात्री", "roads", "P3"],
 ["बोअरवेल चालू आहे पण गटाराचं पाणी mix होतंय", "water_supply", "P1"],
 ["sewage treatment plant काम करत नाही smell येतो दूरपर्यंत", "drainage", "P3"],
 ["construction site वरचा debris रस्त्यावर ठेवलंय हटवा", "solid_waste", "P3"],
 ["light pole jvळ wire open aahe danger aahe", "street_lights", "P1"],
 ["मोठं झाड रस्त्याच्या मध्यावर आहे traffic ला problem", "trees_garden", "P3"],
 ["unauthorized dhaba चालवतोय residential area मध्ये", "encroachment", "P3"],
 ["cockroach chi problem aahe building madhe pest control karaa", "health_mosquito", "P3"],
 ["कचरा आणि drainage problem दोन्ही एकत्र सोडवा", "other", "P3"],
 ["amchya lane madhe rasta nahi muddy aahe puri cement kara", "roads", "P3"],
 ["fire hydrant ला पाणी नाही emergency ला problem होईल", "water_supply", "P2"],
 ["manholeचं झाकण गायब आहे रात्री कोणी पाहू शकत नाही", "drainage", "P1"],
 ["medical waste dustbin मध्ये टाकतात segregation नाही", "solid_waste", "P2"],
 ["compound wall तुटून पडलीय streetlight pole वाकलंय", "street_lights", "P2"],
 ["park maintenance budget कमी आहे सगळं खराब होतंय", "trees_garden", "P4"],
 ["godown बांधलंय residential zone मध्ये permission नसताना", "encroachment", "P3"],
 ["rat menace area madhe PCMC action ghya urgently", "health_mosquito", "P2"],
 ["एक problem सांगता दुसरं निघतं सगळंच बिघडलंय area मध्ये", "other", "P4"],
 ["हॉस्पिटलजवळ open drain आहे patients ला infection होतोय", "drainage", "P1"],
 ["शाळेजवळ कचरा जळवतात मुलांना श्वसनाचा त्रास", "solid_waste", "P2"],
 ["electrical wire sagging aahe road var touch hoil", "street_lights", "P1"],
 ["bridge chi railing tutli aahe koni padel river madhe", "roads", "P1"],
 ["public toilet madhe paani nahi sanitation problem", "health_mosquito", "P3"],
 ["building chi bhint padayla aali aahe crack aahet saglikade", "encroachment", "P1"],
 ["vacant plot var kacharyacha dhig laglay rats yetaat udeer", "solid_waste", "P3"],
 ["नवीन area मध्ये streetlight connection नाही अजून", "street_lights", "P3"],
 ["garden ला lock केलंय कोणी उघडत नाही", "trees_garden", "P4"],
 ["hawker zone बाहेर stalls लावतात traffic jam होतो", "encroachment", "P3"],
 ["dengue chi case vadhle aahet area madhe fogging lavkar karaa", "health_mosquito", "P2"],
 ["सगळं problem आहे area मध्ये रस्ते पाणी light सगळं", "other", "P3"],
 ["pothole madhe bike skid jhali accident zhalay minor", "roads", "P2"],
 ["nळ तुटलाय water supply बंद आहे colony साठी", "water_supply", "P2"],
 ["पावसाळ्यात basement मध्ये drainage चं पाणी येतं", "drainage", "P2"],
 ["रस्त्याच्या कडेला कचरा टाकतात नदीत जातो pollution", "solid_waste", "P3"],
 ["transformer blast zhalay area madhe current nahi urgent", "street_lights", "P1"],
 ["park madhla jogging track tutala aahe stones niklya", "trees_garden", "P3"],
 ["balkani todun extra room bandhlay permission nahi illegal", "encroachment", "P3"],
 ["jaundice pasarla aahe contaminated water supply mule", "health_mosquito", "P1"],
 ["मी तक्रार करतो पण काहीच होत नाही कोणी ऐकत नाही", "other", "P4"],
 ["sagla kahi problem aahe amchya bhagat rasta paani kachra light", "other", "P3"],
 ["दवाखान्याजवळ रस्ता खचलाय ambulance अडकते", "roads", "P2"],
 ["borewell cha paani band aahe tanker pathva urgently", "water_supply", "P2"],
 ["नाल्यात chemical waste सोडतात factory मधून", "drainage", "P2"],
 ["कचऱ्याचं decomposition होतंय गरम वातावरणात दुर्गंधी unbearable", "solid_waste", "P3"],
 ["तारा तुटल्यात विजेच्या खांबावरून पडायला आल्यात", "street_lights", "P1"],
 ["garden madhe snake aala aahe mulanna dhoka aahe", "trees_garden", "P2"],
 ["unauthorized parking lot banvla aahe government jaagevar", "encroachment", "P3"],
 ["water borne disease pasarla aahe colony madhe check karaa", "health_mosquito", "P2"],
 ["काय complaint करू कशाबद्दल करू सगळंच बिघडलंय", "other", "P4"],
 ["rasta todla drainage sathi parat kela nahi aata khadde aahet", "other", "P3"]
]

df_a = pd.DataFrame([(t, d, "A", None) for d, ts in DATA.items() for t in ts], columns=["text", "department", "source", "severity_label"])
df_b = pd.DataFrame([(t, d, "B", sv) for t, d, sv in DATA_B], columns=["text", "department", "source", "severity_label"])
df = pd.concat([df_a, df_b], ignore_index=True)
df.to_csv(f"{OUT_DIR}/nagarvani_seed_dataset_AB.csv", index=False)
print(df.shape); print(pd.crosstab(df.department, df.source))

## Step 2 — M3 Normalisation (light)
Unicode NFC, remove zero-width characters, collapse spaces, lowercase Latin text.
*Viva note:* the full M3 (transliteration + civic lexicon) is planned; character n-grams in Step 3 already absorb some spelling variation.

In [ ]:
def normalise(s: str) -> str:
    s = unicodedata.normalize("NFC", s or "")
    s = s.replace("\u200c", "").replace("\u200d", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

X = df.text.map(normalise).tolist()
y = np.array(df.department.tolist())
SRC = df.source.values
print(X[:3])

## Step 3 — M4a Department classifier + baseline
- **Baseline:** keyword matching (count department keywords, pick the max).
- **Learned model:** TF-IDF on character 2–5-grams + logistic regression. Character n-grams suit Marathi because words inflect heavily (रस्त्यावर, रस्त्याचे, रस्ता share रस्त).
- **Hybrid (used in the demo):** the same n-grams plus 8 civic-lexicon features (keyword hits per department), learned jointly. This is a first version of the PBL-3 M3 civic lexicon feeding M4a.
- **Evaluation:** 5-fold stratified cross-validation on both sets combined (9 classes including *Other*).

*Viva note:* the civic lexicon was written from **Set A only**. So the cross-validation score here is somewhat optimistic; Step 3b gives the honest held-out number.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

KEYWORDS = {
    "roads": ["खड्ड", "रस्त", "रोड", "road", "rasta", "फुटपाथ", "footpath", "pothole", "khadd", "सड़क", "गड्ढ", "डांबर"],
    "water_supply": ["नळ", "पाणी", "paani", "pani", "पानी", "nal", "टँकर", "tanker", "पाईपलाईन", "pipeline", "pressure", "दाब"],
    "drainage": ["ड्रेनेज", "drainage", "गटार", "gutter", "चेंबर", "chamber", "मॅनहोल", "manhole", "नाल", "सांडपाणी", "मैनहोल", "नाली"],
    "solid_waste": ["कचर", "kachr", "garbage", "कूड़", "घंटागाडी", "dustbin", "कचरे", "राडारोडा", "प्लास्टिक"],
    "street_lights": ["दिव", "light", "लाइट", "खांब", "khamb", "अंधार", "andhar", "andhera", "बल्ब", "खंभ"],
    "trees_garden": ["झाड", "jhaad", "jhad", "फांदी", "phandi", "उद्यान", "बाग", "garden", "park", "पार्क", "पेड़", "ped ", "tree"],
    "encroachment": ["अतिक्रमण", "फेरीवाल", "feriwal", "बांधकाम", "bandhkam", "construction", "कब्जा", "kabja", "होर्डिंग", "hoarding", "अवैध", "अनधिकृत", "बेकायदेशीर", "इमारत"],
    "health_mosquito": ["डास", "dasa", "मच्छर", "fogging", "फवारणी", "डेंग्यू", "dengue", "कुत्र", "kutr", "कुत्त", "शौचालय", "toilet", "उंद", "मलेरिया"],
}
def keyword_baseline(text):
    scores = {d: sum(k in text for k in ks) for d, ks in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "roads"   # arbitrary fallback

base_pred = np.array([keyword_baseline(t) for t in X])

from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
KW_DEPTS = sorted(KEYWORDS)
def keyword_features(texts):   # 8 numbers per complaint: keyword hits per department (civic lexicon)
    return 0.5 * np.array([[sum(k in t for k in KEYWORDS[d]) for d in KW_DEPTS] for t in texts], dtype=float)

tfidf_only = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    ("clf", LogisticRegression(C=10, max_iter=3000)),
])
model = Pipeline([   # hybrid: learned character n-grams + civic-lexicon features
    ("features", FeatureUnion([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
        ("lexicon", FunctionTransformer(keyword_features)),
    ])),
    ("clf", LogisticRegression(C=10, max_iter=3000)),
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
CLASSES = np.array(sorted(set(y)))
tfidf_pred = CLASSES[cross_val_predict(tfidf_only, X, y, cv=skf, method="predict_proba").argmax(1)]
oof_proba = cross_val_predict(model, X, y, cv=skf, method="predict_proba")
oof_pred = CLASSES[oof_proba.argmax(1)]

rows = [("Keyword rules only* (cannot predict Other)", base_pred), ("TF-IDF char n-grams + LogReg", tfidf_pred),
        ("Hybrid: TF-IDF + lexicon features + LogReg*", oof_pred)]
results = pd.DataFrame([(n, accuracy_score(y, p), f1_score(y, p, average="macro")) for n, p in rows],
                       columns=["model", "accuracy", "macro_F1"]).round(3)
print("PRELIMINARY results, 5-fold CV on Sets A+B combined\n"); print(results.to_string(index=False))
print("\n* the keyword lexicon was written from Set A, which is part of this data, so these rows are optimistic.")
print("  Step 3b gives the held-out number.")
print("\nHybrid model, per class:\n", classification_report(y, oof_pred, digits=3))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y, oof_pred, labels=CLASSES, xticks_rotation=45, ax=ax, colorbar=False)
ax.set_title("Hybrid classifier — confusion matrix (5-fold CV, seed data)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150); plt.show()

## Step 3b — Held-out check: train on one set, test on the other
Sets A and B were drafted separately, in different styles. Training on A and testing on B (and the reverse) scores the model on sentences written independently of its training data, and of the lexicon (which came from A). *Other* is left out because Set A has no *Other* class.

*Viva note:* **this** is the number to quote as our honest accuracy, not the cross-validation score above.

In [ ]:
from sklearn.base import clone
k8 = df.department.values != "other"
Xa = [x for x, s, k in zip(X, SRC, k8) if s == "A" and k]; ya = y[(SRC == "A") & k8]
Xb = [x for x, s, k in zip(X, SRC, k8) if s == "B" and k]; yb = y[(SRC == "B") & k8]
held = []
for name, m in [("TF-IDF char n-grams + LogReg", tfidf_only), ("Hybrid (lexicon from Set A)", model)]:
    pb = clone(m).fit(Xa, ya).predict(Xb); pa = clone(m).fit(Xb, yb).predict(Xa)
    held.append((name, accuracy_score(yb, pb), f1_score(yb, pb, average="macro"),
                 accuracy_score(ya, pa), f1_score(ya, pa, average="macro")))
held = pd.DataFrame(held, columns=["model", "A→B acc", "A→B macro-F1", "B→A acc", "B→A macro-F1"]).round(3)
print("HELD-OUT (train on one set, test on the other; 8 departments)\n"); print(held.to_string(index=False))
print("\nB→A for the hybrid is optimistic (lexicon came from A). A→B is the cleanest number.")

mb = clone(model).fit(Xa, ya); pb = mb.predict(Xb)
scripts = df[(df.source == "B") & k8].text.map(lambda t: "romanised" if not re.search("[\u0900-\u097F]", t) else ("code-mixed" if re.search("[A-Za-z]", t) else "devanagari")).values
print("\nA→B accuracy by script (hybrid):")
for sc in ["devanagari", "code-mixed", "romanised"]:
    kk = scripts == sc; print(f"  {sc:11s} n={kk.sum():3d}  acc {(pb[kk] == yb[kk]).mean():.3f}")

## Step 4 — M7 Abstention threshold from data
For each threshold τ: complaints with top probability ≥ τ are routed automatically, the rest go to a human.
We plot **review rate** (share sent to humans) against **misrouting rate** (share of auto-routed complaints sent to the wrong department) and pick the lowest review rate with misrouting ≤ 5%.

*Viva note:* this is the "abstention is an output, not a failure" principle from PBL-3, chosen from out-of-fold predictions rather than guessed. Caveat to state: on the held-out set (A→B) misrouting is higher than 5% at this τ, because probabilities are not yet calibrated. Calibration (temperature scaling) is planned.

In [ ]:
pmax = oof_proba.max(1)
correct = oof_pred == y
taus = np.round(np.arange(0.0, 1.0, 0.01), 2)
review_rate, misroute_rate = [], []
for t in taus:
    routed = pmax >= t
    review_rate.append(1 - routed.mean())
    misroute_rate.append(((~correct) & routed).sum() / max(routed.sum(), 1))
review_rate, misroute_rate = np.array(review_rate), np.array(misroute_rate)

TARGET_MISROUTE = 0.05
ok = np.where(misroute_rate <= TARGET_MISROUTE)[0]
TAU = float(taus[ok[0]]) if len(ok) else 0.5
i = int(np.where(taus == TAU)[0][0])
print(f"Chosen TAU = {TAU}: review rate {review_rate[i]:.1%}, misrouting {misroute_rate[i]:.1%} (seed data, preliminary)")

plt.figure(figsize=(6, 4.5))
plt.plot(review_rate * 100, misroute_rate * 100, marker=".")
plt.scatter([review_rate[i] * 100], [misroute_rate[i] * 100], s=80, zorder=3, label=f"chosen τ = {TAU}")
plt.xlabel("Sent to human review (%)"); plt.ylabel("Misrouted among auto-routed (%)")
plt.title("Abstention trade-off (5-fold CV, Sets A+B)"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/tau_curve.png", dpi=150); plt.show()

pa_b = mb.predict_proba(Xb).max(1); routed_b = pa_b >= TAU
print(f"Held-out A→B at the same τ: review {1 - routed_b.mean():.1%}, misrouting {((pb != yb) & routed_b).sum() / max(routed_b.sum(), 1):.1%}")

model.fit(X, y)   # final model on both sets for the demo

## Step 5 — M5 Severity: life-safety rules + department defaults
- Life-safety patterns **force P1** regardless of the model (PBL-3: deterministic rules override learned models where safety is at stake).
- Otherwise the department default band applies, raised one level by urgency cues (accident, school, hospital, danger, weeks without action).
- Rules can list words that cancel them, e.g. झाडला ("swept") must not trigger the fallen-tree rule just because it contains झाड.
- Every decision carries a human-readable reason.
- SLA hours (P1 24 h, P2 3 days, P3 7 days, P4 15 days) are **our proposed values**, not official PMC figures.

In [ ]:
LIFE_SAFETY = [  # (reason, any-of terms, and-any-of terms, none-of terms); empty second list = first list alone is enough
    ("open manhole / उघडे मॅनहोल", ["मॅनहोल", "manhole", "मैनहोल", "चेंबर", "chamber"], ["झाकण", "उघड", "cover", "गायब", "ढक्कन", "तुट", "open", "missing"], []),
    ("missing drain cover / गटाराचे झाकण", ["गटार", "drainage", "drain"], ["झाकण", "cover", "ढक्कन"], []),
    ("live or fallen wire / लोंबकळणारी वायर", ["वायर", "wire", "तार", "cable"], ["लोंबक", "लटकत", "latkat", "तुट", "tutl", "पडल", "padl", "padal", "गिर", "sagging", "open", "danger", "शॉक", "shock"], []),
    ("electric shock / विजेचा धक्का", ["शॉक", "shock", "करंट लाग", "करंट येत", "current lag", "current yet", "electrocut"], [], []),
    ("fallen or falling tree / पडलेले झाड", ["झाड", "jhaad", "jhad", "zaad", "पेड़", "फांदी", "phandi"], ["पडले", "पडला", "पडली", "पडू", "पडेल", "पडायला", "padla", "padli", "padu", "padel", "गिर"], ["झाडला", "झाडले जात", "झाडलं जात", "झाडू", "jhadla", "jhadu", "कचरा", "waste"]),
    ("collapse risk / कोसळण्याचा धोका", ["कोसळ", "collapse", "गिर सकता", "पडायला आल", "padayla aal"], [], []),
    ("broken bridge railing / पुलाचा कठडा", ["railing", "रेलिंग", "कठडा"], ["तुट", "tutl", "broken", "पडेल", "padel"], []),
    ("contaminated drinking water / दूषित पाणी", ["contaminat", "sanitary", "विहिरीत", "गटाराचं पाणी", "गटारीचं पाणी"], ["पिण्याच", "drinking", "विहिर", "बोअरवेल", "borewell", "water", "मिसळ", "mix", "आजारी"], []),
    ("disease outbreak / साथ", ["outbreak", "jaundice", "कावीळ", "cholera", "कॉलरा"], [], []),
    ("fire hazard / आग", ["fire exit", "आग लाग"], [], []),
    ("sewage entering homes / घरात सांडपाणी", ["सांडपाणी", "sandpaani", "sewage", "ड्रेनेज", "drainage", "गटार", "gutter"], ["घरात", "ghar", "शिरत"], []),
    ("injury or animal bite / जखम किंवा चावा", ["जखमी", "चावले", "चावतात", "chavtat", "इजा", "पिसाळ", "wala padla", "मुलं पडत"], [], []),
]
URGENCY_CUES = ["अपघात", "accident", "शाळ", "school", "हॉस्पिटल", "hospital", "रुग्णालय", "धोका", "dhoka", "धोकादायक", "खतरनाक",
                "khatarnak", "लहान मुल", "मुलांना", "urgent", "लवकर", "lavkar", "आठवड", "हफ्ते", "महिन"]
DEFAULT_BAND = {"drainage": "P2", "water_supply": "P2", "roads": "P3", "solid_waste": "P3", "street_lights": "P3",
                "trees_garden": "P3", "health_mosquito": "P3", "encroachment": "P4", "other": "P3"}
SLA_HOURS = {"P1": 24, "P2": 72, "P3": 168, "P4": 360}

def severity(text, dept):
    for reason, first, second, exclude in LIFE_SAFETY:
        if (any(a in text for a in first) and (not second or any(b in text for b in second))
                and not any(x in text for x in exclude)):
            return "P1", [f"life-safety rule: {reason}"]
    band = DEFAULT_BAND.get(dept, "P3"); reasons = [f"department default: {band}"]
    cue = next((c for c in URGENCY_CUES if c in text), None)
    if cue and band != "P2":
        band = f"P{int(band[1]) - 1}"; reasons.append(f"urgency cue '{cue}' raised it to {band}")
    return band, reasons

for t in ["मॅनहोल उघडे आहे", "कचरा गाडी आली नाही", "शाळेसमोर कचरा आहे"]:
    print(t, "→", severity(normalise(t), "drainage" if "मॅनहोल" in t else "solid_waste"))

# Check against Set B's severity labels. The rules were revised after reading Set B, so treat P1 recall as optimistic.
b = df[df.source == "B"]
pred_band = np.array([severity(normalise(t), d)[0] for t, d in zip(b.text, b.department)])
lab = b.severity_label.values
print(f"\nSet B: P1 recall {(pred_band[lab == 'P1'] == 'P1').mean():.2f} ({(lab == 'P1').sum()} life-safety complaints), "
      f"false P1 {((pred_band == 'P1') & (lab != 'P1')).sum()} of {(lab != 'P1').sum()}, exact band match {(pred_band == lab).mean():.2f}")
print(pd.crosstab(pd.Series(lab, name="label"), pd.Series(pred_band, name="predicted")))

## Step 6 — M2 Speech recognition (Whisper large-v3, zero-shot Marathi)
Confidence = mean word probability. If Whisper thinks there was no speech, the transcript is discarded and the citizen is asked to re-record. Confidence is capped at 0.2 if Whisper repeats one word over and over, and if the output contains no Devanagari at all (Whisper sometimes translates to English or invents text on silence). Below `ASR_TAU` the complaint goes to human review with a note to re-record. Any ASR error returns an empty transcript instead of crashing the app.
If the GPU runs out of memory, change `"large-v3"` to `"medium"`.

In [ ]:
import torch, whisper
ASR_MODEL_NAME = "large-v3"   # change to "medium" if the GPU runs out of memory
asr_model = whisper.load_model(ASR_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

def transcribe(path):
    # Returns (text, confidence). Never raises: any failure returns ("", 0.0) and the app asks the citizen to re-record or type.
    try:
        r = asr_model.transcribe(path, language="mr", task="transcribe", word_timestamps=True,
                                 fp16=torch.cuda.is_available())
    except Exception as e:
        print("ASR error:", e)
        return "", 0.0
    segs = r.get("segments", [])
    if not segs or all(seg.get("no_speech_prob", 0) > 0.6 for seg in segs):
        return "", 0.0          # silence or noise: Whisper may invent text here, so discard it
    words = [w for seg in segs for w in seg.get("words", [])]
    conf = float(np.mean([w["probability"] for w in words])) if words else 0.0
    text = r["text"].strip()
    if text and not re.search("[\u0900-\u097F]", text):
        conf = min(conf, 0.2)   # no Devanagari at all: Whisper probably translated or hallucinated
    toks = text.split()
    if len(toks) >= 6 and len(set(toks)) / len(toks) < 0.3:
        conf = min(conf, 0.2)   # Whisper stuck repeating one word, a known failure on noise
    return text, conf
print("ASR ready:", ASR_MODEL_NAME)

## Step 7 — The triage pipeline (M3 → M4a → M5 → M7) and ticket log

In [ ]:
TICKETS = []

def triage(text, ward, asr_conf=1.0, source="typed"):
    norm = normalise(text)
    proba = model.predict_proba([norm])[0]
    classes = model.classes_
    order = proba.argsort()[::-1]
    dept, p1 = classes[order[0]], float(proba[order[0]])
    margin = p1 - float(proba[order[1]])
    top3 = [(classes[j], float(proba[j])) for j in order[:3]]
    band, reasons = severity(norm, dept)

    review = []
    if sum(ch.isalpha() for ch in norm) < MIN_LETTERS: review.append("no usable words to classify")
    if dept == "other": review.append("classifier says 'Other': no department fits")
    if asr_conf < ASR_TAU: review.append(f"low speech-recognition confidence ({asr_conf:.2f}); ask citizen to re-record")
    if p1 < TAU: review.append(f"classifier unsure (p = {p1:.2f} < τ = {TAU})")
    if margin < 0.10: review.append(f"top two departments too close (margin {margin:.2f})")
    status = "Routed" if not review else "Human review"
    now = dt.datetime.now(IST)
    t = {
        "ticket": uuid.uuid4().hex[:8].upper(),
        "time": now.strftime("%d %b %H:%M:%S"),
        "created": now,
        "ward": ward,
        "status": status,
        "department": DEPT_LABEL[dept] if status == "Routed" else "— (officer to assign)",
        "suggested": " | ".join(f"{d} {p:.2f}" for d, p in top3),
        "priority": band,
        "sla_due": (now + dt.timedelta(hours=SLA_HOURS[band])).strftime("%d %b %H:%M"),
        "confidence": round(min(asr_conf, p1), 2),
        "reasons": "; ".join(reasons + review),
        "source": source,
        "text": text,
    }
    # a P1 in review still sorts to the top
    TICKETS.append(t)
    return t, top3, review

TABLE_COLS = ["ticket", "time", "ward", "status", "department", "priority", "sla_due", "confidence", "suggested", "reasons", "text"]
def tickets_df():
    if not TICKETS: return pd.DataFrame(columns=TABLE_COLS)
    d = pd.DataFrame(TICKETS)
    return d.sort_values(["priority", "created"], ascending=[True, False])[TABLE_COLS]   # P1 first, then newest

# quick check with typed complaints (no audio needed)
for s in ["मॅनहोल उघडे आहे, रात्री कोणी पडू शकते", "तीन दिवसांपासून कचरा उचलला नाही",
          "nalala paani yet nahi", "इथे खूप त्रास आहे, कोणीतरी बघा"]:
    t, top3, rev = triage(s, "Kothrud-Bavdhan")
    print(f"{s}\n  → {t['status']} | {t['department']} | {t['priority']} | {t['reasons']}\n")
TICKETS.clear()

## Step 7b — Self-check (run before every demo)
Checks the demo sentences, life-safety sentences (must be P1), ordinary complaints (must not be P1) and junk input (must not crash, must go to a human). It prints PASS or FAIL for each and never stops the notebook, so the app below still launches.

*Viva note:* "We test the core logic every time the notebook runs." Known, deliberate behaviour: a sentence like "the manhole cover has been fixed, thanks" is still flagged P1. For safety rules we prefer a false alarm to a missed danger.

In [ ]:
CHECKS = [  # (text, expected status, expected department key or None, expected priority)
    ("कोथरूड मध्ये रस्त्यावर मोठा खड्डा आहे गाड्यांना त्रास होतोय", "Routed", "roads", "P3"),
    ("गटार तुंबलंय पाणी रस्त्यावर वाहतंय दुर्गंधी", "Routed", "drainage", "P2"),
    ("मॅनहोल उघडे आहे, रात्री कोणी पडू शकते", "Routed", "drainage", "P1"),
    ("इथे खूप त्रास आहे, कोणीतरी बघा", "Human review", None, None),
    ("मॅनहोल उघडा आहे रस्त्यावर कोणी पडणार", "Human review", None, "P1"),
    ("nalala don divsapasun pani aala nahi", "Routed", "water_supply", "P2"),
    # edge cases that used to fail
    ("light nahi current nahi aahe", None, None, "not P1"),
    ("रस्ता झाडला जात नाही, सगळीकडे प्लास्टिक पडले आहे", None, None, "not P1"),
    ("मुलाला करंट लागला", None, None, "P1"),
    ("wire padli aahe road var", None, None, "P1"),
    ("zaad padla aahe rastyavar", None, None, "P1"),
    ("रोज accident होत आहेत", None, None, "not P1"),
    # life-safety sentences must always be P1
    ("wire tutli aahe khamb var latkat aahe", None, None, "P1"),
    ("झाड विजेच्या तारांवर पडले आहे", None, None, "P1"),
    ("गटाराचं झाकण तुटलंय कोणी पडणार", None, None, "P1"),
    ("इमारतीचा भाग कोसळण्याच्या स्थितीत आहे", None, None, "P1"),
    ("पिसाळलेला कुत्रा परिसरात फिरत आहे", None, None, "P1"),
    ("सांडपाणी घरात शिरत आहे", None, None, "P1"),
    ("open manhole on the road near school", None, None, "P1"),
    # ordinary complaints must never be P1
    ("कचरा गाडी दोन दिवस आली नाही", None, None, "not P1"),
    ("पथदिवा बंद आहे", None, None, "not P1"),
    ("नळाला पाणी कमी दाबाने येते", None, None, "not P1"),
    ("फुटपाथवर फेरीवाले बसतात", None, None, "not P1"),
    ("उद्यानातील बाकडे तुटले आहेत", None, None, "not P1"),
    # junk input: must not crash, must go to a human
    ("", "Human review", None, None),
    ("   ", "Human review", None, None),
    ("!!!", "Human review", None, None),
    ("12345", "Human review", None, None),
    ("🙂🙂", "Human review", None, None),
    ("रस्त्यावर खड्डा " * 60, None, None, None),                  # very long: must not crash
    ("there is a big pothole near my house", None, None, None),   # English: must not crash
]
n_fail = 0
for text, st, dep, pr in CHECKS:
    try:
        t, top3, _ = triage(text, "Kothrud-Bavdhan")
        ok = (st is None or t["status"] == st) and (dep is None or top3[0][0] == dep) and \
             (pr is None or (t["priority"] != "P1" if pr == "not P1" else t["priority"] == pr))
        got = f"{t['status']}, {top3[0][0]}, {t['priority']}"
    except Exception as e:
        ok, got = False, f"CRASH {type(e).__name__}: {e}"
    n_fail += not ok
    print(("PASS " if ok else "FAIL ") + f"{text[:45]!r:49} → {got}")
TICKETS.clear()   # self-check tickets must not appear in the demo table
print("\nALL CHECKS PASSED" if n_fail == 0 else f"\n{n_fail} CHECK(S) FAILED: read the lines above before demoing")

## Step 8 — M1 Web page (Gradio)
Record or upload a complaint (or type it), pick the ward, submit. The ticket table below shows the ward queue sorted by priority.

In [ ]:
import gradio as gr

PRIORITY_TEXT = {"P1": "P1 — तातडीचे / Urgent (24 h)", "P2": "P2 — उच्च / High (3 days)",
                 "P3": "P3 — सामान्य / Normal (7 days)", "P4": "P4 — कमी / Low (15 days)"}

def handle(audio_path, typed, ward):
    try:
        return _handle(audio_path, typed, ward)
    except Exception as e:                    # the page must never show a raw error
        print("App error:", repr(e))
        return ("", f"**काहीतरी चुकले / Something went wrong ({type(e).__name__}). कृपया तक्रार टाइप करा / Please type the complaint.**", tickets_df(), None)

def _handle(audio_path, typed, ward):
    if not ward:
        return ("", "**कृपया वॉर्ड निवडा / Please select a ward.**", tickets_df(), None)
    transcript, asr_conf, source = "", 1.0, "typed"
    if audio_path:
        try:
            transcript, asr_conf = transcribe(audio_path); text, source = transcript, "voice"
        except Exception as e:
            return ("", f"**ऑडिओ प्रक्रिया करता आला नाही / Could not process the audio.** Please record again or type the complaint. ({type(e).__name__})", tickets_df(), None)
    else:
        text = (typed or "").strip()
    if not re.search(r"[A-Za-z\u0900-\u097F]", text or ""):
        return (transcript, "**तक्रार ऐकू आली नाही / No complaint detected. Please record again or type it.**", tickets_df(), None)
    now = dt.datetime.now(IST)
    recent = [x for x in TICKETS if x["ward"] == ward and normalise(x["text"]) == normalise(text)
              and (now - x["created"]).total_seconds() < 120]
    if recent:                                # double-click / accidental resubmission guard
        return (transcript, f"**ही तक्रार आधीच नोंदवली आहे / Already received as ticket `{recent[-1]['ticket']}`.**", tickets_df(), None)
    t, top3, review = triage(text, ward, asr_conf, source)
    lines = [f"### Ticket `{t['ticket']}` — **{t['status']}**",
             f"- **Department:** {t['department']}",
             f"- **Priority:** {PRIORITY_TEXT[t['priority']]} (SLA due {t['sla_due']})",
             f"- **Ward office:** {ward}",
             f"- **Confidence:** {t['confidence']}" + (f" (speech {asr_conf:.2f})" if source == "voice" else ""),
             "- **Top-3 departments:** " + ", ".join(f"{DEPT_LABEL[d]} ({p:.2f})" for d, p in top3),
             f"- **Why:** {t['reasons']}"]
    if review:
        lines.append("\n> तुमची तक्रार अधिकाऱ्याकडे तपासणीसाठी पाठवली आहे. / Sent to an officer for review — it is not lost.")
    path = f"{OUT_DIR}/nagarvani_tickets.csv"; tickets_df().to_csv(path, index=False)
    return transcript, "\n".join(lines), tickets_df(), path

with gr.Blocks(title="NagarVani") as demo:
    gr.Markdown("# NagarVani — नागरी तक्रार (prototype)\nतुमची तक्रार मराठीत बोला किंवा लिहा. Speak or type your complaint in Marathi.")
    with gr.Row():
        with gr.Column():
            audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="🎙 तक्रार बोला / Record complaint")
            typed = gr.Textbox(label="किंवा लिहा / Or type (ignored if audio is recorded)", lines=2)
            ward = gr.Dropdown(WARDS, label="वॉर्ड कार्यालय / Ward office")
            btn = gr.Button("पाठवा / Submit", variant="primary")
        with gr.Column():
            transcript = gr.Textbox(label="ऐकलेला मजकूर / Transcript", interactive=False)
            result = gr.Markdown()
    gr.Markdown("## Ward queue (sorted by priority)")
    table = gr.Dataframe(value=tickets_df(), interactive=False, wrap=True)
    csv_file = gr.File(label="Export tickets (CSV)")
    btn.click(handle, [audio, typed, ward], [transcript, result, table, csv_file])

demo.launch(share=True, debug=False)